### Generative adversarial network

## Data

In [ ]:
import pandas as pd
from PIL import Image
import io
import matplotlib.pyplot as plt
import re
from typing import List
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df=pd.read_parquet("../../datos/faithful_1.21.1_32x32.parquet")
print(device)

img_size=32
embed_dim = 128
latent_dim = 256

## Code

informacion relevante:
las imagenes componen valores entre [-1,1] por eso se usa funcion tanh y no sigmoid

In [ ]:
def mostrar_resultados(resultados):
    """
    Muestra una lista de imágenes generadas con sus etiquetas.
    """
    plt.figure(figsize=(8, 2))
    for i, (img, label) in enumerate(resultados):
        plt.subplot(1, len(resultados), i+1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(label)
    plt.tight_layout()
    plt.show()

def mostrar_metricas(epocas, valores, num_epochs, label:str=""):
    plt.figure(figsize=(6, 2))
    plt.xlim(0, num_epochs)
    # Ajuste dinámico del límite Y basado en el histórico
    max_val = max(valores)
    plt.ylim(0, max_val * 1.1)
    
    plt.xlabel('Epoca')
    plt.ylabel(label)
    plt.plot(epocas, valores, color="r", marker='o', linewidth=1, label=label)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def generar_imagenes(G, text_embedding, prompt, n_samples=4, latent_dim=latent_dim, device=device, return_pil = True):
    """Genera n_samples del promt si es único y si es lista de promts, genera 1 por cada promt"""

    if isinstance(prompt, str):
        lista_prompts = [prompt] * n_samples
    else:
        lista_prompts = prompt
        n_samples = len(lista_prompts)

    G.eval()
    text_embedding.eval()
    with torch.no_grad():
        # Embedding condicional del texto
        cond_emb = text_embedding(lista_prompts).to(device)

        # Ruido aleatorio
        z = torch.randn(n_samples, latent_dim, device=device)

        # Generar imágenes
        x = G(z, cond_emb).detach().cpu()

        # Desnormalizar
        x = (x * 0.5 + 0.5).clamp(0, 1)

        # Convertir a PIL y devolver
        if return_pil: 
            return [(TF.to_pil_image(img.cpu()), p) for img, p in zip(x, lista_prompts)] 
        # Devolver tensores pytorch
        return x


def total_variation_loss(img): 
    """introducir penalizacion a que las imagenes no sean continuas"""
    diff_i = torch.mean(torch.abs(img[:, :, 1:, :] - img[:, :, :-1, :]))
    diff_j = torch.mean(torch.abs(img[:, :, :, 1:] - img[:, :, :, :-1]))
    return diff_i + diff_j


# Dataset único (con augment)
class MinecraftDataset(Dataset):
    def __init__(self, rows: List[dict], image_size=img_size, augment=False):
        self.rows = rows
        self.image_size = image_size
        self.augment = augment
        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)])

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        img = Image.open(io.BytesIO(r["image"]["bytes"])).convert("RGB")
        img_t = self.transform(img)
        if self.augment and torch.rand(1).item() > 0.5:
            img_t = torch.flip(img_t, dims=[2])
        label = r["label"]
        return img_t, label


# Embedding textual (por palabra)
class TextEmbedding(nn.Module):
    def __init__(self, vocab, embed_dim=100):
        super().__init__()
        self.vocab = vocab
        self.stoi = {w: i for i, w in enumerate(vocab)}
        self.embedding = nn.Embedding(len(vocab), embed_dim)

    @staticmethod
    def build_vocab(labels):
        words = set()
        for lbl in labels:
            for w in re.findall(r"\w+", lbl.lower()):
                words.add(w)
        return sorted(words)

    def forward(self, label_texts: List[str]):
        device = next(self.embedding.parameters()).device
        vectors = []
        for text in label_texts:
            tokens = [w for w in re.findall(r"\w+", text.lower()) if w in self.stoi]
            if not tokens:
                tokens = ["unknown"]
            idxs = torch.tensor([self.stoi[w] for w in tokens], device=device)
            emb = self.embedding(idxs).mean(dim=0)
            vectors.append(emb)
        return torch.stack(vectors)


#### Crear Modelo

In [ ]:
# Generator (3 capas + salida, CONCATENACION)
class Generator(nn.Module):
    def __init__(self, embed_dim=embed_dim, latent_dim=latent_dim, base_ch=32, out_channels=3):
        super().__init__()
        # le entran un vector de ruido y de condicionamineto del mismo tamaño concatenados
        # Proyección inicial → 4×4
        self.flat_dim = base_ch * 8 * 4 * 4
        self.fc = nn.Linear(latent_dim + embed_dim, self.flat_dim)

        self.net = nn.Sequential(
            # Bloque de Profundidad Adicional (4x4)
            nn.Conv2d(base_ch * 8, base_ch * 8, 3, 1, 1, bias=False),  # Mantiene 4x4, 8*base_ch canales
            nn.BatchNorm2d(base_ch * 8),
            nn.ReLU(True),
            
            # 4×4 → 8×8 (Upsample)
            nn.ConvTranspose2d(base_ch * 8, base_ch * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_ch * 4),
            nn.ReLU(True),

            # Bloque de Profundidad Adicional (8x8)
            nn.Conv2d(base_ch * 4, base_ch * 4, 3, 1, 1, bias=False), # Mantiene 8x8, 4*base_ch canales
            nn.BatchNorm2d(base_ch * 4),
            nn.ReLU(True),

            # 8×8 → 16×16 (Upsample)
            nn.ConvTranspose2d(base_ch * 4, base_ch * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_ch * 2),
            nn.ReLU(True),

            # Bloque de Profundidad Adicional (16x16)
            nn.Conv2d(base_ch * 2, base_ch * 2, 3, 1, 1, bias=False), # Mantiene 16x16, 2*base_ch canales
            nn.BatchNorm2d(base_ch * 2),
            nn.ReLU(True),

            # Capa adicional (16×16 → 32×32) (Upsample)
            nn.ConvTranspose2d(base_ch * 2, base_ch, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(True),

            # Bloque de Profundidad Adicional (32x32)
            nn.Conv2d(base_ch, base_ch, 3, 1, 1, bias=False), # Mantiene 32x32, base_ch canales
            nn.BatchNorm2d(base_ch),
            nn.ReLU(True),

            # Salida (32×32)
            nn.ConvTranspose2d(base_ch, out_channels, 3, 1, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z, cond_emb):
        x = torch.cat([z, cond_emb], dim=1)
        x = self.fc(x)
        x = x.view(x.size(0), -1, 4, 4)
        return self.net(x)

# Discriminator
class Discriminator(nn.Module):
    def __init__(self, embed_dim=embed_dim, base_ch=32, in_channels=3):
        super().__init__()
        self.fc_cond = nn.Linear(embed_dim, img_size * img_size)  # cond_map → 32×32

        self.net = nn.Sequential(
            # 4 canales → 32×32 → 16×16 (Downsample)
            nn.Conv2d(in_channels + 1, base_ch, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # Bloque de Profundidad Adicional (16x16)
            nn.Conv2d(base_ch, base_ch, 3, 1, 1, bias=False), # Mantiene 16x16, base_ch canales
            nn.BatchNorm2d(base_ch),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 16×16 → 8×8 (Downsample)
            nn.Conv2d(base_ch, base_ch * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_ch * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # Bloque de Profundidad Adicional (8x8)
            nn.Conv2d(base_ch * 2, base_ch * 2, 3, 1, 1, bias=False), # Mantiene 8x8, 2*base_ch canales
            nn.BatchNorm2d(base_ch * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # 8×8 → 4×4 (Downsample)
            nn.Conv2d(base_ch * 2, base_ch * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_ch * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # Bloque de Profundidad Adicional (4x4)
            nn.Conv2d(base_ch * 4, base_ch * 4, 3, 1, 1, bias=False), # Mantiene 4x4, 4*base_ch canales
            nn.BatchNorm2d(base_ch * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # salida (4×4 → escalar)
            nn.Flatten(),
            nn.Linear(base_ch * 4 * 4 * 4, 1),
            nn.Sigmoid()
        )

    def forward(self, img, cond_emb):
        B, C, H, W = img.shape
        cond_map = self.fc_cond(cond_emb).view(B, 1, H, W)
        x = torch.cat([img, cond_map], dim=1)
        return self.net(x).view(-1)


In [ ]:
# === CONFIGURACIÓN ===
base_ch = 64

batch_size = 128
lr_G = 2e-4
lr_D = 1e-4

# parametros del generador
lambda_tv = 0.1  # intensidad de la correccion del suavizado  menor que antes (antes 0.1)
lambda_l2 = 0.2 # peso del L2  grande para que la recreación de imágenes sea más precisa.


In [ ]:
# Construir vocabulario
all_labels = df["label"].tolist()
cleaned_labels = [re.sub(r"\d+", "", lbl).strip() for lbl in all_labels]
all_labels = list(set(cleaned_labels))

vocab = TextEmbedding.build_vocab(all_labels)

print(f" Vocabulario generado con {len(vocab)} palabras únicas:")
# print(vocab)

# Dataset y DataLoader
dataset = MinecraftDataset(df.to_dict("records"), image_size=img_size, augment=False)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


#### Entrenamiento

In [ ]:
# MODELOS
txt_emb = TextEmbedding(vocab, embed_dim=embed_dim).to(device)
G = Generator(embed_dim=embed_dim, latent_dim=latent_dim, base_ch=base_ch).to(device)
D = Discriminator(embed_dim=embed_dim, base_ch=base_ch).to(device)


In [ ]:
# Pérdida y optimizadores
criterion = nn.BCELoss()
optimizerG = optim.Adam(G.parameters(), lr=lr_G)
optimizerT = optim.Adam(txt_emb.parameters(), lr=lr_G) # Para el text embedder
optimizerD = optim.Adam(D.parameters(), lr=lr_D)


In [ ]:
# entrenamiento
num_epochs = 4000
# Listas para guardar el historial
historial_epocas = []
historial_lossD = []
historial_lossG = []

for epoch in range(num_epochs):
    G.train()
    D.train()
    txt_emb.train()
    lossD_epoch = 0
    lossG_epoch = 0
    for real_imgs, labels in dataloader:
        real_imgs = real_imgs.to(device)
        cond_emb = txt_emb(labels).to(device)
        b_size = real_imgs.size(0)

        # Etiquetas reales/falsas con label smoothing leve
        real_labels = torch.full((b_size,), 0.9, dtype=torch.float, device=device)
        fake_labels = torch.full((b_size,), 0.0, dtype=torch.float, device=device)

        # =========================================================
        # (1) Actualizar Discriminator: maximize log(D(x|y)) + log(1 - D(G(z|y)))
        # =========================================================
        optimizerD.zero_grad()
        optimizerT.zero_grad()

        # imgs falsas 
        z = torch.randn(b_size, latent_dim, device=device)
        fake_imgs = G(z, cond_emb.detach())

        # Salida del D con imágenes reales
        output_real = D(real_imgs, cond_emb)
        lossD_real = criterion(output_real, real_labels)

        # Salida del D con imágenes falsas
        output_fake = D(fake_imgs.detach(), cond_emb.detach())
        lossD_fake = criterion(output_fake, fake_labels)

        # Pérdida total del Discriminador
        lossD = (lossD_real + lossD_fake) / 2
        lossD.backward()
        optimizerD.step()
        optimizerT.step()

        # =========================================================
        # (2) Actualizar Generator: maximize log(D(G(z|y)))
        # =========================================================
        optimizerG.zero_grad()
        optimizerT.zero_grad()

        cond_emb_G = txt_emb(labels) #necesitamos ejecutarlo otra vez para poder propagar gradientes
        fake_imgs_G = G(z, cond_emb_G)
        
        out_fake_G = D(fake_imgs_G, cond_emb_G)
        lossG_adv = criterion(out_fake_G, real_labels)  # pérdida adversarial
    
        # tv_loss = total_variation_loss(fake_imgs) # añadir suavidad
        # loss_pixel = F.mse_loss(fake_imgs, real_imgs)  # comparación directa de píxeles

        lossG = lossG_adv # + lambda_tv * tv_loss + lambda_l2 * loss_pixel
        lossG.backward()
        optimizerG.step()
        optimizerT.step()

        #Errores
        lossD_epoch += lossD.item()
        lossG_epoch += lossG.item()

    # =========================================================
    # LOGGING
    # =========================================================
    if epoch % 100 == 0:
        test_prompt = "iron boat"
        historial_epocas.append(epoch)
        historial_lossD.append(lossD_epoch / len(dataloader))
        historial_lossG.append(lossG_epoch / len(dataloader))

        print(f"Epoch [{epoch}/{num_epochs}] ")
        print(f"Generator Loss: {lossG_epoch / len(dataloader):.4f} | Discriminator Loss: {lossD_epoch / len(dataloader):.4f}")

        mostrar_metricas(historial_epocas, historial_lossD, num_epochs, "loss Discriminator")
        mostrar_metricas(historial_epocas, historial_lossG, num_epochs, "loss Generator")
        
        # Generar una imagen de ejemplo condicional
        mostrar_resultados(generar_imagenes(G, txt_emb, test_prompt))


#### Guardar modelos

In [ ]:
# GUARDAR MODELOS
import os
import torch

version = "vf1" 
save_dir = f"./modelos/{version}"

os.makedirs(save_dir, exist_ok=True)

# === Guardar modelos ===
torch.save(G.state_dict(), os.path.join(save_dir, "generator.pth"))
torch.save(D.state_dict(), os.path.join(save_dir, "discriminator.pth"))
torch.save(txt_emb.state_dict(), os.path.join(save_dir, "text_embedding.pth"))

# guardamos hiperparámetros y vocabulario
metadata = {
    "embed_dim": embed_dim,
    "latent_dim": latent_dim,
    "base_ch": base_ch,
    "vocab": txt_emb.vocab,}

torch.save(metadata, os.path.join(save_dir, "metadata.pth"))

print(f" Modelos guardados en: {save_dir}")


#### resultados

In [ ]:
# Generar y mostrar
prompt = "golden door"
resultados = generar_imagenes(G, txt_emb, prompt)
mostrar_resultados(resultados)

In [ ]:
prompt = "emerald pickaxe"
imgs = generar_imagenes(G, txt_emb, prompt)
mostrar_resultados(imgs)


In [ ]:
prompt = "redstone ingot"
imgs = generar_imagenes(G, txt_emb, prompt)
mostrar_resultados(imgs)

In [ ]:
prompt = "blue slime dye"
imgs = generar_imagenes(G, txt_emb, prompt)
mostrar_resultados(imgs)

In [ ]:
prompt = "rotten flesh"
imgs = generar_imagenes(G, txt_emb, prompt)
mostrar_resultados(imgs)

#### Importar modelos

In [ ]:
import torch
import os

# Configuración
device = "cuda" if torch.cuda.is_available() else "cpu"
version = "vf1"  
save_dir = f"./modelos/{version}"

metadata = torch.load(os.path.join(save_dir, "metadata.pth"))
txt_emb = TextEmbedding(vocab=metadata["vocab"], embed_dim=metadata["embed_dim"]).to(device)
G = Generator(embed_dim=metadata["embed_dim"], latent_dim=metadata["latent_dim"], base_ch=metadata["base_ch"]).to(device)
D = Discriminator(embed_dim=metadata["embed_dim"], base_ch=metadata["base_ch"]).to(device)
latent_dim = metadata["latent_dim"] #necesario para la validación de metricas

# === Cargar pesos ===
G.load_state_dict(torch.load(os.path.join(save_dir, "generator.pth"), map_location=device))
D.load_state_dict(torch.load(os.path.join(save_dir, "discriminator.pth"), map_location=device))
txt_emb.load_state_dict(torch.load(os.path.join(save_dir, "text_embedding.pth"), map_location=device))

print(" Modelos cargados correctamente desde", save_dir)


#### Validación de métricas de resultados

In [ ]:
prompts_inventados = [
    "bamboo sword",
    "glass pickaxe",
    "magma boat",
    "slime boots",
    "copper apple",
    "phantom shield",
    "amethyst door",
    "netherite bone",
    "rotten cake",
    "spider bow",
    "wooden chestplate",
    "baked melon",
    "glowstone helmet",
    "snow sword",
    "crimson water",
    "iron bamboo",
    "diamond chicken",
    "emerald fish",
    "enchanted clay",
    "turtle axe",
    "golden paper",
    "lapis apple",
    "beef pie",
    "quartz boat",
    "redstone sword",
    "leather compass",
    "honeycomb shield",
    "chorus bread",
    "bone compass",
    "poisonous cake"
]

In [ ]:
import torch
import gc
import random
from transformers import CLIPProcessor, CLIPModel, BlipProcessor, BlipForImageTextRetrieval
import transformers
import random

def generar_prompts_aleatorios(txt_emb, n_samples=4, palabras_por_prompt=2):
    # Filtramos la palabra "unknown" para que no aparezca en las combinaciones
    vocab_limpio = [w for w in txt_emb.vocab if w != "unknown"]
    prompts_generados = [] 

    for _ in range(n_samples):
        # Elegimos palabras al azar con reemplazo
        palabras_elegidas = random.choices(vocab_limpio, k=palabras_por_prompt)
        prompt = " ".join(palabras_elegidas)
        prompts_generados.append(prompt)

    return prompts_generados

def evaluar_clip_blip(model, txt_emb, n=200, batch_size=64, img_size=img_size, device=device):
    # Cargamos los modelos de evaluacion
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    blip_model = BlipForImageTextRetrieval.from_pretrained("Salesforce/blip-itm-base-coco").to(device)
    blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-itm-base-coco")
    
    clip_model.eval()
    blip_model.eval()
    
    clip_score_total = 0.0
    blip_score_total = 0.0
    conteo = 0
    
    
    # Iteramos directamente sobre la cantidad n
    for i in range(0, n, batch_size):
        actual_n = min(batch_size, n - i)
        
        # Generamos solo la cantidad necesaria para este lote
        lote_textos = prompts_inventados
        
        # Generamos las imagenes llamando a la funcion externa
        resultados_gen = generar_imagenes(
            model, 
            txt_emb, 
            prompt=lote_textos, 
            latent_dim = latent_dim,
            device=device, 
            return_pil=True
        )
        
        # Extraemos solo las imagenes PIL
        imagenes_pil = [res[0] for res in resultados_gen]
        
        with torch.no_grad():
            # Calcular CLIP Score
            inputs_clip = clip_processor(text=lote_textos, images=imagenes_pil, return_tensors="pt", padding=True).to(device)
            outputs_clip = clip_model(**inputs_clip)
            image_embeds = outputs_clip.image_embeds
            text_embeds = outputs_clip.text_embeds

            image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
            text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

            clip_scores = (image_embeds * text_embeds).sum(dim=-1)

            clip_score_total += clip_scores.sum().item()
                    
            # Calcular BLIP Score
            inputs_blip = blip_processor(text=lote_textos, images=imagenes_pil, return_tensors="pt", padding=True).to(device)
            outputs_blip = blip_model(**inputs_blip)
            
            # Aplicamos softmax para obtener la probabilidad real de que el texto coincida con la imagen (indice 1)
            itm_probs = torch.nn.functional.softmax(outputs_blip.itm_score, dim=1)
            blip_score_total += itm_probs[:, 1].sum().item()
        
        conteo += actual_n
        
        # Limpieza de memoria
        torch.cuda.empty_cache()
        gc.collect()

    return {
        "CLIP_Score": clip_score_total / conteo,
        "BLIP_Score": blip_score_total / conteo
    }

for i in range(10):
    print(evaluar_clip_blip(G, txt_emb))

#### Tiempo de inferencia medio

In [ ]:
import time
import torch
import numpy as np

def medir_latencia_estadistica_texto(model, txt_emb, n_iterations=1000,  device=device):
    """
    Calcula la media y desviación estándar del tiempo de inferencia para generar 
    una única imagen condicionada por un texto aleatorio distinto cada vez (Batch Size = 1).
    """
    model.eval()
    txt_emb.eval()
    tiempos = []
    
    print(f"Iniciando medición estadística sobre {n_iterations} iteraciones con prompts aleatorios...")

    with torch.no_grad():
        # Fase de calentamiento (Warm-up)
        # Generamos un prompt cualquiera para calentar la red
        prompt_warmup = generar_prompts_aleatorios(txt_emb, n_samples=1)
        
        for _ in range(5):
            _ = generar_imagenes(model, txt_emb, prompt=prompt_warmup, latent_dim = latent_dim, device=device, return_pil=False)
        
        if device == "cuda":
            torch.cuda.synchronize()

        # Ciclo de medición individual
        for i in range(n_iterations):
            # Generamos un nuevo prompt aleatorio exclusivo para esta iteración
            prompt_actual = generar_prompts_aleatorios(txt_emb, n_samples=1)
            
            if device == "cuda":
                torch.cuda.synchronize() # Sincronización previa
            
            t_inicio = time.perf_counter()
            
            # Llamada a la función de generación con el texto nuevo
            _ = generar_imagenes(model, txt_emb, prompt=prompt_actual, device=device, return_pil=False)
            
            if device == "cuda":
                torch.cuda.synchronize() # Sincronización posterior
            
            t_fin = time.perf_counter()
            tiempos.append(t_fin - t_inicio)

    # Cálculo de métricas descriptivas
    tiempos_arr = np.array(tiempos)
    mean_time = np.mean(tiempos_arr)
    std_time = np.std(tiempos_arr)
    
    print(f"Resultados de Inferencia (n=1 por iteración)")
    print(f"mean: {mean_time:.6f} s, std: {std_time:.6f} s")

    return {
        "mean": mean_time,
        "std": std_time
    }


medir_latencia_estadistica_texto(G, txt_emb)